# Running NKI kernels on CPU with no Trainium hardware

This notebook shows how to install and use the **NKI CPU simulator** on a plain Ubuntu
instance (laptop, dev box, CI runner, or any EC2 instance) that has **no Trainium or
Inferentia device**. The simulator runs NKI kernels on CPU via NumPy, so you can develop
and correctness-test kernels before ever touching a Trn/Inf instance.

**What it is good for**
- Correctness iteration while writing a kernel (fast: seconds per launch).
- Debugging shape / dtype / tile-size / SBUF-PSUM constraint violations — you get the same
  errors you'd get on device.
- CI unit tests for kernel correctness, with no accelerator in the runner.

**What it is NOT for**
- Performance measurement. The wall-clock here is NumPy on CPU, not Trainium cycles.
- Final sign-off. Simulator-green is a development milestone; always validate correctness
  and measure performance on real hardware before shipping.

This notebook covers a tiny hand-written kernel first, then downloads and runs one of our
**published HuggingFace Kernel Hub** kernels (`jburtoft/gelu-erf-neuron`) on the simulator.

Official docs: <https://awsdocs-neuron.readthedocs-hosted.com/en/latest/nki/guides/nki_simulator.html>

## 1. Prerequisites

- Ubuntu (or any Linux) — **no Neuron driver, runtime, or device required**.
- **Python 3.10, 3.11, 3.12, or 3.13.** The NKI wheels on the Neuron pip index ship for
  `cp310`–`cp313` only. Python 3.9 and 3.14 have **no** NKI wheel.

Check your Python version:

In [ ]:
import sys
print(sys.version)
assert (3, 10) <= sys.version_info[:2] <= (3, 13), (
    'NKI wheels are published for Python 3.10-3.13 only. '
    'Create a venv with one of those versions.'
)

If your default Python is outside 3.10–3.13, create a venv first (shell, outside this notebook):

```bash
sudo apt-get update && sudo apt-get install -y python3.12 python3.12-venv
python3.12 -m venv ~/nki-sim
source ~/nki-sim/bin/activate
pip install jupyter ipykernel
python -m ipykernel install --user --name nki-sim --display-name 'Python (nki-sim)'
```

then start Jupyter and select the **Python (nki-sim)** kernel for this notebook.

## 2. Install the `nki` package from the Neuron pip index

The simulator ships inside the standalone `nki` wheel. Install it from the Neuron pip index
(the copy on public PyPI is an empty `0.0.1` placeholder — do **not** install that one).
`ml_dtypes` and `numpy` come in as dependencies and give the simulator hardware-like
`bfloat16` / `float8` behavior. The simulator does **not** need `neuronx-cc` — it is pure NumPy.

In [ ]:
import sys
!{sys.executable} -m pip install \
    --index-url https://pip.repos.neuron.amazonaws.com \
    --extra-index-url https://pypi.org/simple/ \
    nki

Verify the install:

In [ ]:
import nki, numpy as np, ml_dtypes
print('nki        :', nki.__version__)
print('numpy      :', np.__version__)
print('ml_dtypes  :', ml_dtypes.__version__)

## 3. Set the simulator target (IMPORTANT on a hardware-less box)

The simulator picks its NeuronCore generation from `NEURON_PLATFORM_TARGET_OVERRIDE`, an
explicit `target=`, or by auto-detecting the chip. **When there is no chip to detect, it
falls back to `trn3` (gen4).** So on a laptop or non-Neuron EC2 instance, every
`nki.simulate(...)` silently targets trn3 unless you say otherwise.

Set the target to match the hardware you will eventually deploy on. **Set it before
importing `torch` / NKI compiler internals.** For FP32 kernels using only generation-agnostic
ops the numeric result is the same either way, but it matters for **FP8** (different max values
across generations) and any generation-specific ISA intrinsic.

| Value | Hardware |
|-------|----------|
| `trn1` or `gen2` | Trn1 / Inf2 (NeuronCore-v2) |
| `trn2` or `gen3` | Trn2 (NeuronCore-v3) |
| `trn3` or `gen4` | Trn3 (NeuronCore-v4) |

In [ ]:
import os
os.environ['NEURON_PLATFORM_TARGET_OVERRIDE'] = 'trn2'  # set to your deployment target

from nki.compiler.target import resolve_target, target_to_nc_version
t = resolve_target()
print(t, 'gen%d' % target_to_nc_version(t))  # expect: trn2 gen3

## 4. Run a hand-written kernel on the simulator

A minimal element-wise add kernel. There is no compile and no device — `nki.simulate` runs the
kernel body as ordinary Python, routing each `nki.language` / `nki.isa` op to a NumPy
implementation.

In [ ]:
import nki
import nki.language as nl
import numpy as np

@nki.jit
def add_kernel(a_ptr, b_ptr):
    a = nl.load(a_ptr)
    b = nl.load(b_ptr)
    result = nl.add(a, b)
    out = nl.ndarray(a_ptr.shape, dtype=a_ptr.dtype, buffer=nl.shared_hbm)
    nl.store(out, value=result)
    return out

a = np.random.rand(128, 512).astype(np.float32)
b = np.random.rand(128, 512).astype(np.float32)

# Run on the CPU simulator
result = nki.simulate(add_kernel)(a, b)

np.testing.assert_allclose(result, a + b, rtol=1e-5)
print('OK:', result.shape, result.dtype)

### The `NKI_SIMULATOR=1` environment variable (no code change)

Instead of wrapping with `nki.simulate(...)`, you can set `NKI_SIMULATOR=1` and run an
existing script that calls `@nki.jit` kernels directly — every kernel then runs on the
simulator with **zero code changes**:

```bash
NKI_SIMULATOR=1 python my_script.py
```

(The env-var form does not currently support JAX; use the explicit `nki.simulate()` API for
JAX workflows.)

## 5. Run a published HuggingFace Kernel Hub kernel on the simulator

You can simulate a real, published NKI kernel without any hardware. Here we use
**[`jburtoft/gelu-erf-neuron`](https://huggingface.co/kernels/jburtoft/gelu-erf-neuron)** — a
fused exact-erf GELU kernel (forward + backward) used by the DINOv3 ViT-L MLP.

For the simulator we only need the kernel **source file**, so we download it directly with
`hf_hub_download(..., repo_type='kernel')`. (The full `get_kernel(...)` loader is for running the
compiled kernel on real Neuron hardware — it needs the `torch.neuron` backend registered, which
a CPU-only box does not have. The simulator does not need it.)

In [ ]:
import sys
!{sys.executable} -m pip install -q huggingface_hub scipy

In [ ]:
from huggingface_hub import hf_hub_download
import importlib.util

# Download just the NKI kernel source from the published kernel-type repo.
src = hf_hub_download(
    'jburtoft/gelu-erf-neuron',
    'build/torch-neuron/nki_kernels/gelu_erf.py',
    repo_type='kernel',
)
print('kernel source:', src)

# Import gelu_fwd from the downloaded file
spec = importlib.util.spec_from_file_location('gelu_erf', src)
gelu_erf = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gelu_erf)
print('loaded:', [n for n in dir(gelu_erf) if n.startswith('gelu')])

In [ ]:
import numpy as np, nki
from scipy.special import erf

def ref_gelu(x):  # exact erf GELU
    return x * 0.5 * (1.0 + erf(x / np.sqrt(2.0)))

x = np.random.randn(256, 768).astype(np.float32)   # [R, C]

# Run the published kernel on the CPU simulator
y = nki.simulate(gelu_erf.gelu_fwd)(x)

r = ref_gelu(x)
cos = float((y.ravel() @ r.ravel()) / (np.linalg.norm(y) * np.linalg.norm(r)))
print('shape   :', y.shape, y.dtype)
print('cos_sim :', round(cos, 8))
print('max_abs :', float(np.max(np.abs(y - r))))
assert cos > 0.9999, 'GELU kernel disagrees with exact-erf reference'

`cos_sim ≈ 1.0` confirms the published kernel computes exact-erf GELU. The small `max_abs`
comes from the simulator's activation-table approximation of `nl.gelu`; on device the kernel
measures `cos_sim 1.000000` in FP32 (see the kernel's docstring). This is exactly the
develop-on-CPU, validate-on-device split described below.

## 6. Two debugging features worth knowing

**Uninitialized-memory detection.** The simulator fills every freshly allocated tile with a
sentinel (`NaN` for float, `4` for int). Because `NaN` propagates, a kernel that reads memory
it never wrote produces `NaN` in the output — so you can assert on it. (On real hardware that
same read returns arbitrary leftover data and may *coincidentally* pass, so this is a genuine
bug-finder.)

**Precise vs high-precision float mode.** By default (`NKI_PRECISE_FP=1`) low-precision dtypes
are stored via `ml_dtypes` for hardware-like numerics. Set `NKI_PRECISE_FP=0` to store
everything as float32: run `=0` first to confirm your kernel's *algorithm* is correct in full
precision, then run `=1` to see how much accuracy the low-precision dtypes cost. This separates
“my logic is wrong” from “my downcasts lose precision.”

In [ ]:
# Uninitialized-memory check: a kernel that only writes the first row leaves the rest NaN.
@nki.jit
def partial_write(a_ptr):
    out = nl.ndarray(a_ptr.shape, dtype=a_ptr.dtype, buffer=nl.shared_hbm)
    row = nl.load(a_ptr[0:1, :])
    nl.store(out[0:1, :], value=row)   # only row 0 is written
    return out

a = np.random.rand(4, 8).astype(np.float32)
res = nki.simulate(partial_write)(a)
print('any NaN (expected True — rows 1-3 never written):', bool(np.any(np.isnan(res))))

## 7. Where to go next

- The simulator does **not** model instruction scheduling, engine parallelism, SBUF/PSUM
  overlap, or DMA ordering. Kernels can pass here and still fail or differ on device.
  Treat a green simulator run as a development checkpoint, not a sign-off.
- For final correctness and **all** performance numbers (latency, throughput, MFU), run on a
  real Trainium/Inferentia instance and profile with `neuron-explorer`.
- `nki.simulate` reference: <https://awsdocs-neuron.readthedocs-hosted.com/en/latest/nki/guides/nki_simulator.html>
- NKI language guide: <https://awsdocs-neuron.readthedocs-hosted.com/en/latest/nki/get-started/nki-language-guide.html>